In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType, DateType, TimestampType
import pyspark.sql.functions as F

In [0]:
catalog_name= 'ecommerce'

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)


+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|        _source_file|         injected_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      NOVW|  NovaWave |           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|     VOLT@|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      COTC| CottonClub|    

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         injected_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|     VOLT@|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         injected_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      VOLT|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-04-01 22:43:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver.select("category_code").distinct().show()


+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
anomalies ={
    "GROCERY": "GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}
df_silver = df_silver.replace(anomalies, subset="category_code")

In [0]:
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|          BKS|
|         GRCY|
|          TOY|
|          SPT|
+-------------+



In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

# Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_categories")
df_bronze.show(10)


+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_file|         injected_at|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          hnk|      Home & Kitchen|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          bpc|Beauty & Personal...|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          bks|               Books|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|         grcy|             Grocery|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          toy|        Toys & Games|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          spt|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|         grcy|             Grocery|dbfs:/Volumes/ec

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
df_duplicates.show()

+-------------+-----+
|category_code|count|
+-------------+-----+
|          app|    2|
|         grcy|    2|
+-------------+-----+



In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
df_silver.show()

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_file|         injected_at|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          hnk|      Home & Kitchen|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          bpc|Beauty & Personal...|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          bks|               Books|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|         grcy|             Grocery|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          toy|        Toys & Games|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          spt|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
+-------------+--------------------+--------------------+--------------------+



In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
df_silver.show()


+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_file|         injected_at|
+-------------+--------------------+--------------------+--------------------+
|           CE|         Electronics|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          APP|             Apparel|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          HNK|      Home & Kitchen|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          BPC|Beauty & Personal...|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          BKS|               Books|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|         GRCY|             Grocery|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          TOY|        Toys & Games|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
|          SPT|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-04-01 22:50:...|
+-------------+--------------------+--------------------+--------------------+



In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_categories")

# Products

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")


df_bronze.show(10)

Row count: 50000
Column count: 16
+-------------+--------------+-------------+----------+------+--------+---------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|   product_id|           sku|category_code|brand_code| color|    size| material|weight_grams|length_cm|width_cm|height_cm|rating_count|file_name|ingest_timestamp|        _source_file|         injected_at|
+-------------+--------------+-------------+----------+------+--------+---------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|2000000000015|STCR-HNK-00001|          hnk|      stcr| White|One-Size|    Coton|        305g|     22,2|    17.1|      6.3|           0|     NULL|            NULL|dbfs:/Volumes/eco...|2026-04-01 22:52:...|
|2000000000022|HMNS-HNK-00002|          hnk|      hmns|Silver|One-Size|    Steel|        682g|     18,2|    12.3|      3.7|           1|     N

In [0]:
display(df_bronze.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp,_source_file,injected_at
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z


In [0]:
#removing the g from the weigh in grams and converting it to integer

df_silver = df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305         |
|682         |
|243         |
|225         |
|455         |
+------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn(
    "length_cm", F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(5, truncate=False)



+---------+
|length_cm|
+---------+
|22.2     |
|18.2     |
|18.2     |
|17.6     |
|27.2     |
+---------+
only showing top 5 rows


In [0]:
df_silver.select("category_code","brand_code").show(2)

+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          hnk|      stcr|
|          hnk|      hmns|
+-------------+----------+
only showing top 2 rows


In [0]:
df_silver = df_silver.withColumn(
    "category_code", F.upper(F.col("category_code"))
)
df_silver = df_silver.withColumn(
    "brand_code", F.upper(F.col("brand_code"))
)
df_silver.select("category_code","brand_code").show(10)


+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          HNK|      STCR|
|          HNK|      HMNS|
|           CE|      NOVW|
|          APP|      URTL|
|         GRCY|      GGRN|
|          BPC|      SLKE|
|           CE|      VOLT|
|          APP|      CBLT|
|          SPT|      ARFT|
|          APP|      MOSA|
+-------------+----------+
only showing top 10 rows


In [0]:
df_silver.show(5)

+-------------+--------------+-------------+----------+------+--------+--------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|   product_id|           sku|category_code|brand_code| color|    size|material|weight_grams|length_cm|width_cm|height_cm|rating_count|file_name|ingest_timestamp|        _source_file|         injected_at|
+-------------+--------------+-------------+----------+------+--------+--------+------------+---------+--------+---------+------------+---------+----------------+--------------------+--------------------+
|2000000000015|STCR-HNK-00001|          HNK|      STCR| White|One-Size|   Coton|         305|     22.2|    17.1|      6.3|           0|     NULL|            NULL|dbfs:/Volumes/eco...|2026-04-01 22:52:...|
|2000000000022|HMNS-HNK-00002|          HNK|      HMNS|Silver|One-Size|   Steel|         682|     18.2|    12.3|      3.7|           1|     NULL|            NULL|dbfs:/Volumes/eco.

In [0]:
df_silver.select("color").distinct().show()

+------+
| color|
+------+
| White|
|Silver|
|Purple|
|  Blue|
|  Navy|
|Yellow|
| Green|
| Beige|
| Black|
|  Pink|
|  Gray|
|   Red|
|  NULL|
+------+



In [0]:
df_silver.select("material").distinct().show()

+---------+
| material|
+---------+
|    Coton|
|    Steel|
|     Wood|
|    Ruber|
|  Plastic|
|Polyester|
|    Glass|
|  Alumium|
|    Paper|
|  Leather|
+---------+



In [0]:
df_silver = df_silver.withColumn(
    "material", 
    F.when (F.col("material")=="Coton","Cotton")
    .when (F.col("material")=="Ruber", "Rubber")
    .when (F.col("material")=="Alumium", "Aluminum")
    .otherwise(F.col("material"))
)

df_silver.select("material").distinct().show()


+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



In [0]:
display(df_silver.limit(5))


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp,_source_file,injected_at
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Cotton,305,22.2,17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682,18.2,12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243,18.2,13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Rubber,225,17.6,4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Rubber,455,27.2,15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw_data/products/products.csv,2026-04-01T22:52:54.406Z


In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
df_silver.select("size").distinct().show()

+--------+
|    size|
+--------+
|One-Size|
|       S|
|      XS|
|      XL|
|       L|
|     XXL|
|       M|
+--------+



In [0]:
df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

+------------+---------+-------------+----------+---------+------------+
|weight_grams|length_cm|category_code|brand_code|material |rating_count|
+------------+---------+-------------+----------+---------+------------+
|305         |22.2     |HNK          |STCR      |Cotton   |0           |
|682         |18.2     |HNK          |HMNS      |Steel    |1           |
|243         |18.2     |CE           |NOVW      |Wood     |0           |
|225         |17.6     |APP          |URTL      |Rubber   |50          |
|455         |27.2     |GRCY         |GGRN      |Rubber   |4           |
|232         |28.0     |BPC          |SLKE      |Plastic  |0           |
|507         |27.2     |CE           |VOLT      |Plastic  |5           |
|261         |27.7     |APP          |CBLT      |Polyester|0           |
|59          |12.5     |SPT          |ARFT      |Plastic  |11          |
|238         |10.7     |APP          |MOSA      |Polyester|6           |
+------------+---------+-------------+----------+--

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschemda","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_products")

# Customers

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(10)

Row count: 300000
Column count: 7
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|       country|state|        _source_file|         injected_at|
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|         India|   MH|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000002|619489725433.0|          AU|     Australia|  VIC|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000003|919390066524.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000004|917073741793.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000005|618478772532.0|          AU|     Australia|   WA|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000006|916441718520.0|          IN|         India|   GJ|dbfs:/Volumes/eco..

In [0]:
df_silver =df_bronze.withColumn("phone", F.regexp_replace(F.col("phone"), r"\D", ""))
df_silver.show(10)

+----------------+-------------+------------+--------------+-----+--------------------+--------------------+
|     customer_id|        phone|country_code|       country|state|        _source_file|         injected_at|
+----------------+-------------+------------+--------------+-----+--------------------+--------------------+
|CUST000000000001|9172800335360|          IN|         India|   MH|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000002|6194897254330|          AU|     Australia|  VIC|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000003|9193900665240|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000004|9170737417930|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000005|6184787725320|          AU|     Australia|   WA|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000006|9164417185200|          IN|         India|   GJ|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000007| 

In [0]:
null_count = df_silver.filter(F.col("customer_id").isNull()).count()
null_count

300

In [0]:
df_silver.filter(F.col("customer_id").isNull()).show(5)

+-----------+-------------+------------+--------------+-----+--------------------+--------------------+
|customer_id|        phone|country_code|       country|state|        _source_file|         injected_at|
+-----------+-------------+------------+--------------+-----+--------------------+--------------------+
|       NULL|9181870435620|          IN|         India|   DL|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|       NULL|9175172430520|          IN|         India|   DL|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|       NULL|         NULL|          IN|         India|   GJ|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|       NULL|4472202146050|          GB|United Kingdom|  WLS|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|       NULL|9169962906320|          IN|         India|   UP|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
+-----------+-------------+------------+--------------+-----+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.dropna(subset=["customer_id"])

row_count= df_silver.count()
print(f"Row count after dropping null values : {row_count}")

Row count after dropping null values : 299700


In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}") 

Number of nulls in phone: 29964


In [0]:
df_silver.filter(F.col("phone").isNull()).show(3)

+----------------+-----+------------+-------+-----+--------------------+--------------------+
|     customer_id|phone|country_code|country|state|        _source_file|         injected_at|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
|CUST000000000007| NULL|          IN|  India|   MH|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000010| NULL|          IN|  India|   RJ|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000026| NULL|          IN|  India|   WB|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
only showing top 3 rows


In [0]:
df_silver = df_silver.fillna("Not Available", subset=["phone"])
df_silver.show(5)


+----------------+-------------+------------+---------+-----+--------------------+--------------------+
|     customer_id|        phone|country_code|  country|state|        _source_file|         injected_at|
+----------------+-------------+------------+---------+-----+--------------------+--------------------+
|CUST000000000001|9172800335360|          IN|    India|   MH|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000002|6194897254330|          AU|Australia|  VIC|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000003|9193900665240|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000004|9170737417930|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
|CUST000000000005|6184787725320|          AU|Australia|   WA|dbfs:/Volumes/eco...|2026-04-01 23:07:...|
+----------------+-------------+------------+---------+-----+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

# Date

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_date")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

Row count: 95
Column count: 7
+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         injected_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|  friday|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|02-08-2025|2025|SATURDAY|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|03-08-2025|2025|  SUNDAY|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
display(df_bronze.limit(5))

date,year,day_name,quarter,week_of_year,_source_file,injected_at
01-08-2025,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
02-08-2025,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
03-08-2025,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
04-08-2025,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
05-08-2025,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z


In [0]:
df_bronze.printSchema()

root
 |-- date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- injected_at: timestamp (nullable = true)



In [0]:
# converting string date to date type
df_silver = df_bronze.withColumn("date", F.to_date(F.col("date"),  "dd-MM-yyyy"))


In [0]:
display(df_silver.limit(5))

date,year,day_name,quarter,week_of_year,_source_file,injected_at
2025-08-01,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
2025-08-02,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
2025-08-03,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
2025-08-04,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z
2025-08-05,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw_data/date/date.csv,2026-04-01T23:08:34.984Z


In [0]:
display(df_silver.select("date").distinct())

date
2025-08-01
2025-08-02
2025-08-03
2025-08-04
2025-08-05
2025-08-06
2025-08-07
2025-08-08
2025-08-09
2025-08-10


In [0]:
duplicates = df_silver.groupBy("date").count().filter(F.col("count") > 1)
print("Total Duplicates : ", duplicates.count())
display(duplicates)


Total Duplicates :  3


date,count
2025-08-29,2
2025-09-25,2
2025-10-13,2


In [0]:
df_silver.select("date").dropDuplicates()

DataFrame[date: date]

In [0]:
df_silver.select("day_name").distinct().show()

+---------+
| day_name|
+---------+
|   friday|
| SATURDAY|
|   SUNDAY|
|   MONDAY|
|  TUESDAY|
|WEDNESDAY|
| thursday|
|wednesday|
|   FRIDAY|
| saturday|
|   sunday|
|  tuesday|
|   monday|
| THURSDAY|
+---------+



In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)


+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         injected_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-02|2025|Saturday|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-03|2025|  Sunday|      3|         -31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-04|2025|  Monday|      3|         -32|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-05|2025| Tuesday|      3|         -32|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
 df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive

df_silver.show(3)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         injected_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|      3|          31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-02|2025|Saturday|      3|          31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-03|2025|  Sunday|      3|          31|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week "), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _source_file|         injected_at|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|Q3-2025|Week 31-2025|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-02|2025|Saturday|Q3-2025|Week 31-2025|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
|2025-08-03|2025|  Sunday|Q3-2025|Week 31-2025|dbfs:/Volumes/eco...|2026-04-01 23:08:...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
#renaming the columns
df_silver = df_silver.withColumnRenamed("week_of_the_year", "week")


In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_date")